In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

## MultiServerMCPClient

사전에 `mcp_server_remote.py` 를 실행해둡니다. 터미널을 열고 가상환경이 활성화 되어 있는 상태에서 서버를 실행해 주세요.

> 명령어
```bash
source .dotenv/script/activate
python mcp_server_remote.py
```

`async with` 로 일시적인 Session 연결을 생성 후 해제

In [10]:
# MultiServerMCPClient를 통해 MCP 서버에서 도구 목록을 받아와서,
# 해당 도구들을 LangGraph의 create_react_agent에 연결하여 에이전트를 생성하고,
# ainvoke_graph를 통해 실제로 에이전트가 도구를 사용할 수 있도록 하는 전체 흐름입니다.
# langchain-mcp-adapters 0.1.0 버전부터 MultiServerMCPClient를 async context manager(`async with`)로 사용할 수 없습니다.
# 공식 메시지에 따르면, 아래와 같이 사용해야 합니다.


from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from utils import ainvoke_graph, astream_graph

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# MCP 서버에 연결하여 도구 목록을 받아옵니다.
client = MultiServerMCPClient(
    {
        "name": {
            "url": "http://localhost:8100/sse",
            "transport": "sse",
        }
    }
)
tools = await client.get_tools()  # MCP에서 도구를 받아옴

# 받아온 도구를 LangGraph 에이전트에 연결
agent = create_react_agent(model, tools)

# 에이전트가 도구를 활용하여 질문에 답변하도록 실행
answer = await ainvoke_graph(agent, {"messages": "Jaeho"})

  + Exception Group Traceback (most recent call last):
  |   File "c:\Users\skyop\jaeho_template\dotenv\Lib\site-packages\IPython\core\interactiveshell.py", line 3697, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "C:\Users\skyop\AppData\Local\Temp\ipykernel_11900\1230278285.py", line 24, in <module>
  |     tools = await client.get_tools()  # MCP에서 도구를 받아옴
  |             ^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "c:\Users\skyop\jaeho_template\dotenv\Lib\site-packages\langchain_mcp_adapters\client.py", line 157, in get_tools
  |     tools_list = await asyncio.gather(*load_mcp_tool_tasks)
  |                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "c:\Users\skyop\jaeho_template\dotenv\Lib\site-packages\langchain_mcp_adapters\tools.py", line 188, in load_mcp_tools
  |     async with create_session(connection) as tool_session:
  |   File "C:\Users\skyop\.pyenv\pyenv-win\versions\3.11.9\Lib\contextlib.py", line 210, in __aenter__
  |     r

In [11]:
await astream_graph(agent, {"messages": "jaeho"})


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
Hello! How can I assist you with "jaeho"? Are you looking for information about a person named Jaeho, or something else related to that term? Please provide more details.

{'node': 'agent',
 'content': AIMessageChunk(content='', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_95d112f245'}, id='run-0b366396-6ebc-4654-affd-b1cab89c61f7'),
 'metadata': {'langgraph_step': 1,
  'langgraph_node': 'agent',
  'langgraph_triggers': ('branch:to:agent', 'start:agent', 'tools'),
  'langgraph_path': ('__pregel_pull', 'agent'),
  'langgraph_checkpoint_ns': 'agent:ce6f467a-02b1-6866-d4fa-e8314f7b4234',
  'checkpoint_ns': 'agent:ce6f467a-02b1-6866-d4fa-e8314f7b4234',
  'ls_provider': 'openai',
  'ls_model_name': 'gpt-4.1-mini',
  'ls_model_type': 'chat',
  'ls_temperature': 0.0}}

In [12]:
# 오류가 나는 이유: langchain-mcp-adapters 0.1.0부터 MultiServerMCPClient는 async context manager(`async with`) 또는 __aenter__를 사용할 수 없습니다.

client = MultiServerMCPClient(
    {
        "name": {
            "url": "http://localhost:8100/sse",
            "transport": "sse",
        }
    }
)
tools = await client.get_tools()  # 올바른 사용법: get_tools()를 await로 직접 호출

print(tools)  # 도구가 표시됨

  + Exception Group Traceback (most recent call last):
  |   File "c:\Users\skyop\jaeho_template\dotenv\Lib\site-packages\IPython\core\interactiveshell.py", line 3697, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "C:\Users\skyop\AppData\Local\Temp\ipykernel_11900\3469553090.py", line 11, in <module>
  |     tools = await client.get_tools()  # 올바른 사용법: get_tools()를 await로 직접 호출
  |             ^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "c:\Users\skyop\jaeho_template\dotenv\Lib\site-packages\langchain_mcp_adapters\client.py", line 157, in get_tools
  |     tools_list = await asyncio.gather(*load_mcp_tool_tasks)
  |                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "c:\Users\skyop\jaeho_template\dotenv\Lib\site-packages\langchain_mcp_adapters\tools.py", line 188, in load_mcp_tools
  |     async with create_session(connection) as tool_session:
  |   File "C:\Users\skyop\.pyenv\pyenv-win\versions\3.11.9\Lib\contextlib.py", line 210, in

In [ ]:
# 에이전트 생성
# client.get_tools()는 coroutine이므로 await로 받아야 함
tools = await client.get_tools()
agent = create_react_agent(model, tools)

In [ ]:
await astream_graph(agent, {"messages": "Jaeho"})


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
Hello! How can I assist you with "Jaeho"? Are you looking for information about a person named Jaeho, or something else related to that name? Please provide more details.

{'node': 'agent',
 'content': AIMessageChunk(content='', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_95d112f245'}, id='run-75615cb2-4258-45ce-986e-bb307038a3d8'),
 'metadata': {'langgraph_step': 1,
  'langgraph_node': 'agent',
  'langgraph_triggers': ('branch:to:agent', 'start:agent', 'tools'),
  'langgraph_path': ('__pregel_pull', 'agent'),
  'langgraph_checkpoint_ns': 'agent:07843cb9-d9dc-2bc0-d639-2cf7d5aabf56',
  'checkpoint_ns': 'agent:07843cb9-d9dc-2bc0-d639-2cf7d5aabf56',
  'ls_provider': 'openai',
  'ls_model_name': 'gpt-4.1-mini',
  'ls_model_type': 'chat',
  'ls_temperature': 0.0}}

## Stdio 통신 방식

Stdio 통신 방식은 로컬 환경에서 사용하기 위해 사용합니다.

- 통신을 위해 표준 입력/출력 사용

참고: 아래의 python 경로는 수정하세요!

In [ ]:
import sys, os, nest_asyncio, asyncio
from pathlib import Path
from mcp.client.stdio import stdio_client, StdioServerParameters
from mcp import ClientSession
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

nest_asyncio.apply()
os.environ.setdefault("PYTHONUNBUFFERED", "1")

async def run_mcp_client_stdio_in_notebook(
    server_file: str = "mcp_server_local(stdio).py",
    user_prompt: str = "Jaeho",
):
    server_path = Path(server_file).resolve()
    errlog_path = server_path.parent / "mcp_stderr.log"
    if not server_path.exists():
        raise FileNotFoundError(f"서버 스크립트 없음: {server_path}")

    if sys.platform == "win32":
        try: asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        except: pass

    server = StdioServerParameters(
        command=sys.executable,
        args=["-u", str(server_path)],
        cwd=str(server_path.parent),
        env=os.environ.copy(),
    )

    try:
        with open(errlog_path, "wb") as err:
            async with stdio_client(server, errlog=err) as (read, write):
                async with ClientSession(read, write) as session:
                    await session.initialize()
                    tools = await load_mcp_tools(session)
                    agent = create_react_agent(ChatOpenAI(model="gpt-4.1-mini", temperature=0), tools)
                    inputs = {"messages": [HumanMessage(content=user_prompt, name="user")]}
                    return [event async for event in agent.astream(inputs)]
    except Exception as e:
        tail = ""
        try:
            if errlog_path.exists():
                data = errlog_path.read_bytes()
                tail = (data[-4096:] if len(data) > 4096 else data).decode("utf-8", "ignore")
        except: pass
        raise RuntimeError(f"STDIO MCP 실행 실패: {e}\n\n=== server stderr (tail) ===\n{tail}") from e


In [ ]:
# === mcp_stdio_helper.py (또는 노트북 첫 셀) ===
import sys, os, asyncio
from pathlib import Path
from contextlib import asynccontextmanager

# 주피터 환경 권장: nest_asyncio
try:
    import nest_asyncio
    nest_asyncio.apply()
except Exception:
    pass

os.environ.setdefault("PYTHONUNBUFFERED", "1")  # 서브프로세스 버퍼링 방지

from mcp.client.stdio import stdio_client, StdioServerParameters
from mcp import ClientSession
from langchain_mcp_adapters.tools import load_mcp_tools

def _win_proactor_policy():
    if sys.platform == "win32":
        try:
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        except Exception:
            pass

@asynccontextmanager
async def start_stdio_mcp(
    server_file: str,
    *,
    python: str = sys.executable,
    cwd: str | None = None,
    env: dict | None = None,
    errlog_path: str | None = "mcp_stderr.log",
):
    """
    STDIO MCP 서버를 서브프로세스로 올리고 ClientSession과 변환된 LangChain tools를 반환합니다.
    사용 예:
        async with start_stdio_mcp("server_stdio.py") as (session, tools):
            ...
    """
    _win_proactor_policy()

    server_path = Path(server_file).resolve()
    if not server_path.exists():
        raise FileNotFoundError(f"서버 스크립트가 없습니다: {server_path}")

    params = StdioServerParameters(
        command=python,
        args=["-u", str(server_path)],            # -u: unbuffered
        cwd=str(cwd or server_path.parent),
        env=(env or os.environ).copy(),
    )

    err_fp = None
    try:
        if errlog_path:
            err_fp = open(errlog_path, "wb")
        async with stdio_client(params, errlog=err_fp) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await load_mcp_tools(session)
                try:
                    yield session, tools
                finally:
                    # 세션 종료는 with 블록 해제 시 자동
                    pass
    except Exception as e:
        # 진단을 돕기 위해 stderr tail 표시
        tail = ""
        try:
            if errlog_path and Path(errlog_path).exists():
                data = Path(errlog_path).read_bytes()
                tail = (data[-4096:] if len(data) > 4096 else data).decode("utf-8", "ignore")
        except Exception:
            pass
        raise RuntimeError(f"STDIO MCP 연결 실패: {e}\n\n=== server stderr (tail) ===\n{tail}") from e
    finally:
        if err_fp:
            err_fp.close()


In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# 1) 서버 올리고 도구 받기
async with start_stdio_mcp("mcp_server_local(stdio).py") as (session, tools):
    # 2) 에이전트 구성 & 실행
    model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    agent = create_react_agent(model, tools)

    inputs = {"messages": [HumanMessage(content="jaeho", name="user")]}
    async for event in agent.astream(inputs):
        print(event)

    # # 한 번에 결과만:
    # result = await agent.ainvoke(inputs)
    # print(result)


{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_vbaUvX6x9sq6aFLN9R1XTeSe', 'function': {'arguments': '{"name": "jaeho"}', 'name': 'get_name'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 53, 'total_tokens': 85, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_6d7dcc9a98', 'id': 'chatcmpl-CKgV82r1jVPNXG5tI9wVAKokrbNRe', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-fec94973-fc3b-49d2-be48-2c6f7afaa36e-0', tool_calls=[{'name': 'get_name', 'args': {'name': 'jaeho'}, 'id': 'call_vbaUvX6x9sq6aFLN9R1XTeSe', 'type': 'tool_call'}], usage_metadata={'input_tokens': 53, 'output_tokens': 32, 'total_tokens': 85, 'input_token_details': {'audio': 0, 'ca

In [ ]:
# 같은 디렉터리에 server_stdio.py 가 있어야 합니다.
# server_stdio.py 안에서는 반드시 로그 print 를 stderr 로 보내세요:
# print("MCP running...", file=sys.stderr)
results = await run_mcp_client_stdio_in_notebook(
    server_file="mcp_server_local(stdio).py",
    user_prompt="Jaeho",
)

for r in results:
    print(r)


{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_rUcPqXsIhdeWJBbQLO3EKuWf', 'function': {'arguments': '{"name": "Jaeho"}', 'name': 'get_name'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 53, 'total_tokens': 85, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_6d7dcc9a98', 'id': 'chatcmpl-CKgVD8tjvXQXicQngwsEERbSiPr5N', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-5e49a007-3521-4356-8783-ce38748a9c3d-0', tool_calls=[{'name': 'get_name', 'args': {'name': 'Jaeho'}, 'id': 'call_rUcPqXsIhdeWJBbQLO3EKuWf', 'type': 'tool_call'}], usage_metadata={'input_tokens': 53, 'output_tokens': 32, 'total_tokens': 85, 'input_token_details': {'audio': 0, 'ca

----------

In [13]:
import sys, os, nest_asyncio, asyncio
from pathlib import Path
from mcp.client.stdio import stdio_client, StdioServerParameters
from mcp import ClientSession
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

nest_asyncio.apply()
os.environ.setdefault("PYTHONUNBUFFERED", "1")

async def run_mcp_client_stdio_in_notebook(
    server_file: str = "mcp_rag_stdio.py",
    user_prompt: str = "Jaeho",
):
    server_path = Path(server_file).resolve()
    errlog_path = server_path.parent / "mcp_stderr.log"
    if not server_path.exists():
        raise FileNotFoundError(f"서버 스크립트 없음: {server_path}")

    if sys.platform == "win32":
        try: asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        except: pass

    server = StdioServerParameters(
        command=sys.executable,
        args=["-u", str(server_path)],
        cwd=str(server_path.parent),
        env=os.environ.copy(),
    )

    try:
        with open(errlog_path, "wb") as err:
            async with stdio_client(server, errlog=err) as (read, write):
                async with ClientSession(read, write) as session:
                    await session.initialize()
                    tools = await load_mcp_tools(session)
                    agent = create_react_agent(ChatOpenAI(model="gpt-4.1-mini", temperature=0), tools)
                    inputs = {"messages": [HumanMessage(content=user_prompt, name="user")]}
                    return [event async for event in agent.astream(inputs)]
    except Exception as e:
        tail = ""
        try:
            if errlog_path.exists():
                data = errlog_path.read_bytes()
                tail = (data[-4096:] if len(data) > 4096 else data).decode("utf-8", "ignore")
        except: pass
        raise RuntimeError(f"STDIO MCP 실행 실패: {e}\n\n=== server stderr (tail) ===\n{tail}") from e


In [14]:
# === mcp_stdio_helper.py (또는 노트북 첫 셀) ===
import sys, os, asyncio
from pathlib import Path
from contextlib import asynccontextmanager

# 주피터 환경 권장: nest_asyncio
try:
    import nest_asyncio
    nest_asyncio.apply()
except Exception:
    pass

os.environ.setdefault("PYTHONUNBUFFERED", "1")  # 서브프로세스 버퍼링 방지

from mcp.client.stdio import stdio_client, StdioServerParameters
from mcp import ClientSession
from langchain_mcp_adapters.tools import load_mcp_tools

def _win_proactor_policy():
    if sys.platform == "win32":
        try:
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        except Exception:
            pass

@asynccontextmanager
async def start_stdio_mcp(
    server_file: str,
    *,
    python: str = sys.executable,
    cwd: str | None = None,
    env: dict | None = None,
    errlog_path: str | None = "mcp_stderr.log",
):
    """
    STDIO MCP 서버를 서브프로세스로 올리고 ClientSession과 변환된 LangChain tools를 반환합니다.
    사용 예:
        async with start_stdio_mcp("server_stdio.py") as (session, tools):
            ...
    """
    _win_proactor_policy()

    server_path = Path(server_file).resolve()
    if not server_path.exists():
        raise FileNotFoundError(f"서버 스크립트가 없습니다: {server_path}")

    params = StdioServerParameters(
        command=python,
        args=["-u", str(server_path)],            # -u: unbuffered
        cwd=str(cwd or server_path.parent),
        env=(env or os.environ).copy(),
    )

    err_fp = None
    try:
        if errlog_path:
            err_fp = open(errlog_path, "wb")
        async with stdio_client(params, errlog=err_fp) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await load_mcp_tools(session)
                try:
                    yield session, tools
                finally:
                    # 세션 종료는 with 블록 해제 시 자동
                    pass
    except Exception as e:
        # 진단을 돕기 위해 stderr tail 표시
        tail = ""
        try:
            if errlog_path and Path(errlog_path).exists():
                data = Path(errlog_path).read_bytes()
                tail = (data[-4096:] if len(data) > 4096 else data).decode("utf-8", "ignore")
        except Exception:
            pass
        raise RuntimeError(f"STDIO MCP 연결 실패: {e}\n\n=== server stderr (tail) ===\n{tail}") from e
    finally:
        if err_fp:
            err_fp.close()


In [16]:
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# 1) 서버 올리고 도구 받기
async with start_stdio_mcp("mcp_rag_stdio.py") as (session, tools):
    # 2) 에이전트 구성 & 실행
    model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    agent = create_react_agent(model, tools)

    inputs = {"messages": [HumanMessage(content="jaeho", name="user")]}
    async for event in agent.astream(inputs):
        print(event)

    # # 한 번에 결과만:
    # result = await agent.ainvoke(inputs)
    # print(result)


{'agent': {'messages': [AIMessage(content='Hello! How can I assist you with "jaeho"? Are you looking for information about a person named Jaeho, or something else related to that name? Please provide more details.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 105, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_95d112f245', 'id': 'chatcmpl-CPB3ajSSJGjc2QDfvf9fIbIP9AlHC', 'finish_reason': 'stop', 'logprobs': None}, id='run-1ccd0aad-56d6-4991-a92e-ccd3c961d940-0', usage_metadata={'input_tokens': 105, 'output_tokens': 40, 'total_tokens': 145, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}}


In [ ]:
# 같은 디렉터리에 server_stdio.py 가 있어야 합니다.
# server_stdio.py 안에서는 반드시 로그 print 를 stderr 로 보내세요:
# print("MCP running...", file=sys.stderr)
results = await run_mcp_client_stdio_in_notebook(
    server_file="mcp_rag_stdio.py",
    user_prompt="DNA 서열 예측",
)

for r in results:
    print(r)


# `mcp_local(stdio).py` 에서 돌려보기

In [ ]:
# from mcp import ClientSession, StdioServerParameters
# from mcp.client.stdio import stdio_client
# from langgraph.prebuilt import create_react_agent
# from langchain_mcp_adapters.tools import load_mcp_tools

# # command 위치가 맞는지 확인하려면, 실제 파이썬 인터프리터 경로가 맞는지 확인해야 합니다.
# # 예를 들어, 윈도우 환경에서는 보통 "python" 또는 "python.exe" 경로가 다를 수 있습니다.
# # 아래처럼 sys.executable을 사용하면 현재 실행 중인 파이썬 경로를 쓸 수 있습니다.

# import sys

# server_params = StdioServerParameters(
#     command=sys.executable,  # 현재 파이썬 인터프리터 경로 사용
#     args=["mcp_local_local(stdio).py"],
# )

# # StdIO 클라이언트를 사용하여 서버와 통신
# async with stdio_client(server_params) as (read, write):
#     # 클라이언트 세션 생성
#     async with ClientSession(read, write) as session:
#         # 연결 초기화
#         await session.initialize()

#         # MCP 도구 로드
#         tools = await load_mcp_tools(session)
#         print(tools)

#         # 에이전트 생성
#         agent = create_react_agent(model, tools)

#         # 에이전트 응답 스트리밍
#         await astream_graph(agent, {"messages": "jaeho"})

# Retriever MCP (mcp_rag_sse.py)

In [9]:
# MultiServerMCPClient를 통해 MCP 서버에서 도구 목록을 받아와서,
# 해당 도구들을 LangGraph의 create_react_agent에 연결하여 에이전트를 생성하고,
# ainvoke_graph를 통해 실제로 에이전트가 도구를 사용할 수 있도록 하는 전체 흐름입니다.
# langchain-mcp-adapters 0.1.0 버전부터 MultiServerMCPClient를 async context manager(`async with`)로 사용할 수 없습니다.
# 공식 메시지에 따르면, 아래와 같이 사용해야 합니다.


from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from utils import ainvoke_graph, astream_graph

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# MCP 서버에 연결하여 도구 목록을 받아옵니다.
client = MultiServerMCPClient(
    {
        "name": {
            "url": "http://localhost:8101/sse",
            "transport": "sse",
        }
    }
)
tools = await client.get_tools()  # MCP에서 도구를 받아옴

# 받아온 도구를 LangGraph 에이전트에 연결
agent = create_react_agent(model, tools)

# 에이전트가 도구를 활용하여 질문에 답변하도록 실행
answer = await ainvoke_graph(agent, {"messages": "spri에 미드저니에 대한 정보가 있어?"})


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  retrieve (call_wtP4mv8bxqF3B7mAv5n0irMZ)
 Call ID: call_wtP4mv8bxqF3B7mAv5n0irMZ
  Args:
    query: 미드저니

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: retrieve

- n AI 이미지 생성 플랫폼 미드저니(Midjourney)가 2025년 6월 19일 비디오 생성 모델 'V1'을 출시
- V1은 이미지를 동영상으로 변환하는 모델로, 미드저니 플랫폼에서 제작된 이미지나 외부 이미지를 바탕으로 동영상을 생성하며, '자동' 설정 시에는 모션 프롬프트가 자동으로 생성되고 '수동' 설정을 선택하면 사용자 지시에 따라 사물이나 장면의 움직임을 생성
- 사용자는 움직임 강도를 피사체와 카메라가 모두 움직이는 '하이 모션(High Motion)'과 카메라는 거의 고정되어 있고 피사체가 천천히 움직이도록 연출 '로우 모션(Low Motion)' 중에서 선택 가능
- 웹 전용 모델인 V1은 1회 동영상 생성 작업으로 5초 길이의 동영상 4개를 제작하며, 생성된 동영상은 한 번에 4초씩 최대 4회까지 영상 길이를 확장할 수 있도록 허용
<!-- image -->

## 기업 ･ 산업

<!-- image -->

## 미드저니, 첫 번째 비디오 생성 AI 모델 'V1' 출시

## KEY Contents

- n 미드저니가 1회 작업으로 5초 길이의 동영상 4개를 제작할 수 있는 

# Manse tool MCP (mcp_manse_sse.py)

In [ ]:
# MultiServerMCPClient를 통해 MCP 서버에서 도구 목록을 받아와서,
# 해당 도구들을 LangGraph의 create_react_agent에 연결하여 에이전트를 생성하고,
# ainvoke_graph를 통해 실제로 에이전트가 도구를 사용할 수 있도록 하는 전체 흐름입니다.
# langchain-mcp-adapters 0.1.0 버전부터 MultiServerMCPClient를 async context manager(`async with`)로 사용할 수 없습니다.
# 공식 메시지에 따르면, 아래와 같이 사용해야 합니다.


from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from utils import ainvoke_graph, astream_graph

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# MCP 서버에 연결하여 도구 목록을 받아옵니다.
client = MultiServerMCPClient(
    {
        "name": {
            "url": "http://localhost:8102/sse",
            "transport": "sse",
        }
    }
)
tools = await client.get_tools()  # MCP에서 도구를 받아옴

# 받아온 도구를 LangGraph 에이전트에 연결
agent = create_react_agent(model, tools)

# 에이전트가 도구를 활용하여 질문에 답변하도록 실행
answer = await ainvoke_graph(agent, {"messages": "1995년 3월 28일 12시 30분 출생 사주봐줘. 자세히 풀이까지 해줘"})


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  calculate_saju (call_V25erY6T9Ph1poW9vTez92DP)
 Call ID: call_V25erY6T9Ph1poW9vTez92DP
  Args:
    year: 1995
    month: 3
    day: 28
    hour: 12
    minute: 30

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: calculate_saju

=== 사주팔자 ===
년주(年柱): 을해
월주(月柱): 기묘
일주(日柱): 무오
시주(時柱): 무오
일간(日干): 무
현재 나이: 30세 / 한국식 나이: 31세
기준 시점: 2025-09-28 16:53:58

=== 오행 강약 (8점 만점) ===
목: 2점
화: 2점
토: 3점
금: 0점
수: 1점

=== 십신 분석 ===
년주: 천간:정관, 지지:편재(70%), 지지:편관(30%)
월주: 천간:겁재, 지지:정관(100%)
일주: 지지:정인(70%), 지지:겁재(30%)
시주: 지지:정인(70%), 지지:겁재(30%)

=== 대운 (정밀 계산) ===
5세: 무인 (2000년 ~ 2009년)
15세: 정축 (2010년 ~ 2019년)
25세: 병자 (2020년 ~ 2029년)
35세: 을해 (2030년 ~ 2039년)

🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
======================

## SSE 방식과 StdIO 방식 혼합 사용

- 파일: `mcp_server_rag.py` 는 StdIO 방식으로 통신
- `langchain-dev-docs` 는 SSE 방식으로 통신
- `manse-tool`은 SSEㅂ 방식으로 통신
SSE 방식과 StdIO 방식을 혼합하여 사용합니다.

`mcp_client_openai.py` 참고

In [ ]:
# import sys
# # 아래 코드는 Jupyter 노트북이 아닌, 터미널(명령 프롬프트, bash 등)에서 실행할 수 있는 Python 파일 예시입니다.
# # 파일명 예시: mcp_multiserver_client.py

# import sys
# import asyncio
# from langchain_mcp_adapters.client import MultiServerMCPClient
# from langgraph.prebuilt import create_react_agent
# from langchain_openai import ChatOpenAI

# async def main():
#     model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

#     # 1. 다중 서버 MCP 클라이언트 생성 (document-retriever는 stdio, langchain-dev-docs는 sse)
#     client = MultiServerMCPClient(
#         {
#             "document-retriever": {
#                 "command": sys.executable,  # 현재 파이썬 실행 파일 경로
#                 "args": ["mcp_server_local(stdio).py"],  # mcp_server_rag.py의 경로를 실제 위치로 수정
#                 "transport": "stdio",
#             },
#             "langchain-dev-docs": {
#                 "url": "https://teddynote.io/mcp/langchain/sse",
#                 "transport": "sse",
#             },
#             "manse-tool":{
#                 "url": "http://localhost:8102/sse",
#                 "transport": "sse"
#             }
#         }
#     )

#     # 2. 도구 목록 받아오기
#     tools = await client.get_tools()

#     # 3. LangGraph 에이전트 생성
#     agent = create_react_agent(model, tools)

#     # 4. 에이전트에게 질문하기 (예시)
#     from utils import ainvoke_graph
#     answer = await ainvoke_graph(agent, {"messages": "1995년 3월 28일 12시 30분 출생 사주봐줘. 자세히 풀이까지 해줘"})
#     print(answer)

# if __name__ == "__main__":
#     asyncio.run(main())

UnsupportedOperation: fileno

# Multi SSE MCP

In [ ]:
# Jupyter 노트북 환경에서는 stdio transport가 정상적으로 동작하지 않습니다.
# (UnsupportedOperation: fileno 에러 발생)
# 따라서 stdio 대신 sse 방식만 사용하거나, stdio는 터미널에서만 사용하세요.

from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.runnables import RunnableConfig

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# document-retriever도 sse 방식으로 예시를 변경 (실제 서버 주소로 수정 필요)
client = MultiServerMCPClient(
    {
        "mcp-rag": {
            "url": "http://localhost:8101/sse",  # 실제 SSE 서버 주소로 변경
            "transport": "sse",
        },
        "langchain-dev-docs": {
            "url": "https://teddynote.io/mcp/langchain/sse",
            "transport": "sse",
        },
        "manse-tool":{
            "url": "http://localhost:8102/sse",
            "transport": "sse"
        }
    }
)

tools = await client.get_tools()

langgraph 의 `create_react_agent` 를 사용하여 에이전트를 생성합니다.

In [9]:
# from langchain_mcp_adapters.client import MultiServerMCPClient
# from langgraph.prebuilt import create_react_agent
# from langchain_openai import ChatOpenAI
# from utils import ainvoke_graph, astream_graph
# from langgraph.checkpoint.memory import MemorySaver
# from langchain_core.runnables import RunnableConfig
# from langchain_openai import ChatOpenAI

# model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

prompt = (
    "You are a smart agent. Answer in Korean.\n"
    "- `mcp-rag` : search SPRI AI docs; cite title/date.\n"
    "- `langchain-dev-docs` : search LangChain/LangGraph docs; return API path + short code.\n"
    "- `manse-tool` : when birth date and time are provided, use it to read Saju (Four Pillars) and return pillars/day-master/five-elements.\n"
)

# 오류 원인: client.get_tools()는 awaitable 객체(코루틴)인데, await 없이 바로 전달해서 발생합니다.
# Jupyter 환경에서는 await를 써야 하므로 아래처럼 수정해야 합니다.
agent = create_react_agent(
    model, tools, prompt=prompt, checkpointer=MemorySaver()
)

In [ ]:
config = RunnableConfig(recursion_limit=30, thread_id=1)
await astream_graph(
    agent,
    {
        "messages": "`retriever`도구를 통해서, 미드저니에 버전에 대해서 설명해줘"
    },
    config=config,
)

In [10]:
config = RunnableConfig(recursion_limit=30, thread_id=1)
await astream_graph(
    agent,
    {
        "messages": "`retriever`도구를 통해서, 미드저니에 버전에 대해서 설명해줘"
    },
    config=config,
)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
- n AI 이미지 생성 플랫폼 미드저니(Midjourney)가 2025년 6월 19일 비디오 생성 모델 'V1'을 출시
- V1은 이미지를 동영상으로 변환하는 모델로, 미드저니 플랫폼에서 제작된 이미지나 외부 이미지를 바탕으로 동영상을 생성하며, '자동' 설정 시에는 모션 프롬프트가 자동으로 생성되고 '수동' 설정을 선택하면 사용자 지시에 따라 사물이나 장면의 움직임을 생성
- 사용자는 움직임 강도를 피사체와 카메라가 모두 움직이는 '하이 모션(High Motion)'과 카메라는 거의 고정되어 있고 피사체가 천천히 움직이도록 연출 '로우 모션(Low Motion)' 중에서 선택 가능
- 웹 전용 모델인 V1은 1회 동영상 생성 작업으로 5초 길이의 동영상 4개를 제작하며, 생성된 동영상은 한 번에 4초씩 최대 4회까지 영상 길이를 확장할 수 있도록 허용
- n 미스트랄 AI(Mistral AI)가 2025년 6월 10일 첫 번째 추론 AI 모델 '마지스트랄(Magistral)'을 출시
- 마지스트랄은 매개변수 240억 개의 오픈소스 버전 '마지스트랄 스몰(Magistral Small)'과 더 강력한 성능의 기업용 버전 '마지스트랄 미디엄(Magistral Medium)'으로 구성
- 독일어, 러시아어, 아랍어, 영어, 이탈리아어, 중국어, 스페인어, 프랑스어 등 다양한 언어로 추론할 수 있으며, 구조화된 계산과 프로그래밍 논리, 규칙 기반 시스템 등 광범위한 기업 활용 사례에 적합
- 마지스트랄 미디엄과 마지스트랄 스몰은 AIME 2024* 벤치마크 평가에서 각각 73.6%와 70.7%를 기록해 딥시크 R1(79.8%)과 같은 경쟁 추론 모델과 비교하면 성능이 다소 떨어지는 것으로 확인 * 2024년 미국 수학 올림피아

{'node': 'agent',
 'content': AIMessageChunk(content='', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_95d112f245'}, id='run-bb6e991d-6716-407d-9f25-c1672022f89a'),
 'metadata': {'thread_id': 1,
  'langgraph_step': 3,
  'langgraph_node': 'agent',
  'langgraph_triggers': ('branch:to:agent', 'start:agent', 'tools'),
  'langgraph_path': ('__pregel_pull', 'agent'),
  'langgraph_checkpoint_ns': 'agent:20927bfb-fe10-bc39-aa0f-8710e28c4714',
  'checkpoint_ns': 'agent:20927bfb-fe10-bc39-aa0f-8710e28c4714',
  'ls_provider': 'openai',
  'ls_model_name': 'gpt-4.1-mini',
  'ls_model_type': 'chat',
  'ls_temperature': 0.0}}

In [ ]:
config = RunnableConfig(recursion_limit=30, thread_id=1)
await astream_graph(
    agent,
    {
        "messages": "`retriever`도구를 통해서, 미드저니에 버전에 대해서 설명해줘"
    },
    config=config,
)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
- n AI 이미지 생성 플랫폼 미드저니(Midjourney)가 2025년 6월 19일 비디오 생성 모델 'V1'을 출시
- V1은 이미지를 동영상으로 변환하는 모델로, 미드저니 플랫폼에서 제작된 이미지나 외부 이미지를 바탕으로 동영상을 생성하며, '자동' 설정 시에는 모션 프롬프트가 자동으로 생성되고 '수동' 설정을 선택하면 사용자 지시에 따라 사물이나 장면의 움직임을 생성
- 사용자는 움직임 강도를 피사체와 카메라가 모두 움직이는 '하이 모션(High Motion)'과 카메라는 거의 고정되어 있고 피사체가 천천히 움직이도록 연출 '로우 모션(Low Motion)' 중에서 선택 가능
- 웹 전용 모델인 V1은 1회 동영상 생성 작업으로 5초 길이의 동영상 4개를 제작하며, 생성된 동영상은 한 번에 4초씩 최대 4회까지 영상 길이를 확장할 수 있도록 허용
- n 미스트랄 AI(Mistral AI)가 2025년 6월 10일 첫 번째 추론 AI 모델 '마지스트랄(Magistral)'을 출시
- 마지스트랄은 매개변수 240억 개의 오픈소스 버전 '마지스트랄 스몰(Magistral Small)'과 더 강력한 성능의 기업용 버전 '마지스트랄 미디엄(Magistral Medium)'으로 구성
- 독일어, 러시아어, 아랍어, 영어, 이탈리아어, 중국어, 스페인어, 프랑스어 등 다양한 언어로 추론할 수 있으며, 구조화된 계산과 프로그래밍 논리, 규칙 기반 시스템 등 광범위한 기업 활용 사례에 적합
- 마지스트랄 미디엄과 마지스트랄 스몰은 AIME 2024* 벤치마크 평가에서 각각 73.6%와 70.7%를 기록해 딥시크 R1(79.8%)과 같은 경쟁 추론 모델과 비교하면 성능이 다소 떨어지는 것으로 확인 * 2024년 미국 수학 올림피아

{'node': 'agent',
 'content': AIMessageChunk(content='', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_4fce0778af'}, id='run-df88b1ad-9245-4bca-9751-93fd46f4ddf0'),
 'metadata': {'thread_id': 1,
  'langgraph_step': 3,
  'langgraph_node': 'agent',
  'langgraph_triggers': ('branch:to:agent', 'start:agent', 'tools'),
  'langgraph_path': ('__pregel_pull', 'agent'),
  'langgraph_checkpoint_ns': 'agent:eb05e83e-7f79-b151-3a87-0b5dbde85a30',
  'checkpoint_ns': 'agent:eb05e83e-7f79-b151-3a87-0b5dbde85a30',
  'ls_provider': 'openai',
  'ls_model_name': 'gpt-4.1-mini',
  'ls_model_type': 'chat',
  'ls_temperature': 0.0}}

In [ ]:
config = RunnableConfig(recursion_limit=30, thread_id=1)
await astream_graph(
    agent,
    {
        "messages": "1995년 3월 28일 12시 30분 출생"
    },
    config=config,
)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
파싱 결과:
{
  "year": 1995,
  "month": 3,
  "day": 28,
  "hour": 12,
  "minute": 30,
  "is_male": true,
  "is_leap_month": false
}
🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
=== 사주팔자 ===
년주(年柱): 을해
월주(月柱): 기묘
일주(日柱): 무오
시주(時柱): 무오
일간(日干): 무
현재 나이: 30세 / 한국식 나이: 31세
기준 시점: 2025-09-28 16:59:07

=== 오행 강약 (8점 만점) ===
목: 2점
화: 2점
토: 3점
금: 0점
수: 1점

=== 십신 분석 ===
년주: 천간:정관, 지지:편재(70%), 지지:편관(30%)
월주: 천간:겁재, 지지:정관(100%)
일주: 지지:정인(70%), 지지:겁재(30%)
시주: 지지:정인(70%), 지지:겁재(30%)

=== 대운 (정밀 계산) ===
5세: 무인 (2000년 ~ 2009년)
15세: 정축 (2010년 ~ 2019년)
25세: 병자 (2020년 ~ 2029년)
35세: 을해 (2030년 ~ 2039년)
🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
1995년 3월 28일 12시 30분에 태어난 남성의 사주팔자는 다음과 같습니다.

- 년주: 을해
- 월주: 기묘
- 일주: 무오
- 시주: 무오
- 일간: 무

오행 강약은 목 2점, 화 2점, 토 3점, 금 0점

{'node': 'agent',
 'content': AIMessageChunk(content='', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_4fce0778af'}, id='run-643846fe-3467-41aa-9669-1ed6ce1d0c33'),
 'metadata': {'thread_id': 1,
  'langgraph_step': 10,
  'langgraph_node': 'agent',
  'langgraph_triggers': ('branch:to:agent', 'start:agent', 'tools'),
  'langgraph_path': ('__pregel_pull', 'agent'),
  'langgraph_checkpoint_ns': 'agent:693bfd80-2512-fac8-4855-765e3862ba6e',
  'checkpoint_ns': 'agent:693bfd80-2512-fac8-4855-765e3862ba6e',
  'ls_provider': 'openai',
  'ls_model_name': 'gpt-4.1-mini',
  'ls_model_type': 'chat',
  'ls_temperature': 0.0}}

In [ ]:
config = RunnableConfig(recursion_limit=30, thread_id=1)
await astream_graph(
    agent,
    {"messages": "langgraph-dev-docs 참고해서 hierarchical-rag 의 정의, 코드에 대해서 알려줘"},
    config=config,
)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
LangGraph 개발 문서에 따르면, hierarchical-rag(계층적 RAG)는 쿼리 분석과 자기 수정(Self-Reflective) RAG를 결합하여 다양한 데이터 소스에서 정보를 계층적으로 검색하고 생성하는 전략입니다. 쿼리 유형에 따라 웹 검색과 인덱스 기반 RAG를 계층적으로 라우팅하여 최적의 답변을 생성합니다.

주요 특징:
- 쿼리 라우팅: 질문을 분석해 웹 검색 또는 벡터스토어 기반 검색으로 분기
- 문서 평가: 검색된 문서의 관련성을 LLM으로 평가
- 질문 재작성: 관련 문서가 없으면 질문을 개선하여 재검색
- 답변 생성: 관련 문서를 바탕으로 답변 생성
- 환각 체크: 생성 답변의 사실성 검증 후 재생성 또는 종료

아래는 LangGraph를 활용한 hierarchical-rag 구현 예시 코드 개요입니다.

```python
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

# 1. 쿼리 라우팅 모델 정의
class RouteQuery(BaseModel):
    datasource: str = Field(..., description="vectorstore or web_search")

llm = ChatOpenAI(model="gpt-4o", temperature=0)
structured_llm_router = llm.with_structured_output(RouteQuery)

system_prompt = """You are an expert at ro

{'node': 'agent',
 'content': AIMessageChunk(content='', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_4fce0778af'}, id='run-b46f8234-1bfc-4e08-8eb6-08f4ee3329b2'),
 'metadata': {'thread_id': 1,
  'langgraph_step': 27,
  'langgraph_node': 'agent',
  'langgraph_triggers': ('branch:to:agent', 'start:agent', 'tools'),
  'langgraph_path': ('__pregel_pull', 'agent'),
  'langgraph_checkpoint_ns': 'agent:cda4e752-cfd4-af6f-f166-b8de7394125b',
  'checkpoint_ns': 'agent:cda4e752-cfd4-af6f-f166-b8de7394125b',
  'ls_provider': 'openai',
  'ls_model_name': 'gpt-4.1-mini',
  'ls_model_type': 'chat',
  'ls_temperature': 0.0}}

## LangChain 에 통합된 도구 + MCP 도구

여기서는 LangChain 에 통합된 도구를 기존의 MCP 로만 이루어진 도구와 함께 사용이 가능한지 테스트 합니다.

In [ ]:
from langchain_tavily import TavilySearch

tavily = TavilySearch(max_result=3, topic="news", days=7)

tools = await client.get_tools()

tools_with_tavily = tools + [tavily]

In [ ]:
tools_with_tavily

[StructuredTool(name='retrieve', description='\n    Retrieves information from the document database based on the query.\n\n    This function creates a retriever, queries it with the provided input,\n    and returns the concatenated content of all retrieved documents.\n\n    Args:\n        query (str): The search query to find relevant information\n\n    Returns:\n        str: Concatenated text content from all retrieved documents\n    ', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'retrieveArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x000001F7D5BAC400>),
 StructuredTool(name='list_of_langchain_documents', description='Retrieves a list of available LangChain and LangGraph documentation resources.\n\n    This function fetches a YAML file from GitHub containing a curated list of\n    documentation sources related to LangCh

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.runnables import RunnableConfig

# 재귀 제한 및 스레드 아이디 설정
config = RunnableConfig(recursion_limit=30, thread_id=3)

# 프롬프트 설정
prompt = "You are a smart agent with various tools. Answer questions in Korean."

# 에이전트 생성
agent = create_react_agent(model, tools_with_tavily, prompt=prompt, checkpointer=MemorySaver())

In [ ]:
# TavilySearch 도구를 통해서,
response = await astream_graph(agent, {"messages": "TavilySearch 도구를 통해서, DeFi Technologies INC 주식의 전망에 대해서 알려줘"}, config=config)
print(response)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
{"query": "DeFi Technologies INC 주식 전망", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://markets.ft.com/data/announce/detail?dockey=600-202509261630PR_NEWS_USPRX____VA84741-1", "title": "DeFi Technologies Announces Closing of US$100 Million Registered Direct Offering - Financial Times", "score": 0.75421005, "published_date": "Fri, 26 Sep 2025 20:30:00 GMT", "content": "DeFi Technologies Inc. (Nasdaq: DEFT) (CBOE CA: DEFI) (GR: R9B) is a financial technology company bridging the gap between traditional capital markets and decentralized finance (\"DeFi\"). DeFi Technologies offers equity investors diversified exposure to the broader decentralized economy through its integrated and scalable business model. This includes Valour, which offers access to digital assets via regulated ETPs; Stillman Digital, a digital ass

## Smithery 에서 제공하는 MCP 서버

- 링크: https://smithery.ai/

사용한 도구 목록은 아래와 같습니다.

- Sequential Thinking: https://smithery.ai/server/@smithery-ai/server-sequential-thinking
  - 구조화된 사고 프로세스를 통해 역동적이고 성찰적인 문제 해결을 위한 도구를 제공하는 MCP 서버
- Desktop Commander: https://smithery.ai/server/@wonderwhy-er/desktop-commander
  - 다양한 편집 기능으로 터미널 명령을 실행하고 파일을 관리하세요. 코딩, 셸 및 터미널, 작업 자동화

**참고**

- smithery 에서 제공하는 도구를 JSON 형식으로 가져올때, 아래의 예시처럼 `"transport": "stdio"` 로 꼭 설정해야 합니다.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

# LLM 모델 초기화
model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# 1. 클라이언트 생성
client = MultiServerMCPClient(
    {
        "server-sequential-thinking": {
            "command": "cmd",
            "args": [
                "/c",
                "npx",
                "-y",
                "@smithery/cli@latest",
                "run",
                "@smithery-ai/server-sequential-thinking",
                "--key",
                "92f8aa68-b8c4-4053-94ee-63d4d0d90992",
                "--profile",
                "allied-bird-lzqFcM"
            ],
            "transport": "stdio"
        },
        "desktop-commander": {
            "command": "cmd",
            "args": [
                "/c",
                "npx",
                "-y",
                "@smithery/cli@latest",
                "run",
                "@wonderwhy-er/desktop-commander",
                "--key",
                "92f8aa68-b8c4-4053-94ee-63d4d0d90992"
            ],
            "transport": "stdio"
        },
        "document-retriever": {
            "command": "./.venv/bin/python",
            # mcp_server_rag.py 파일의 절대 경로로 업데이트해야 합니다
            "args": ["./mcp_server_rag.py"],
            # stdio 방식으로 통신 (표준 입출력 사용)
            "transport": "stdio",
        },
    }
)

# 2. 명시적으로 연결 초기화 (context manager 사용 불가, 공식 가이드에 따라 아래처럼 사용)
# await client.__aenter__()  # 이 방식은 더 이상 지원되지 않음

# 아래와 같이 도구를 불러오세요.
tools = await client.get_tools()

UnsupportedOperation: fileno